# ТЗ для маркетингового исследования. 2ГИС.

- Автор: Бышин М.И. (hh.ru/resume/3a27e4d7ff0ef40ab90039ed1f6f445a567078)

In [1]:
# Импортируем библиотеки
import pandas as pd
import sqlite3

## Задание № 1
- Написать SQL-запрос для решения задачи (синтаксис SQL любой):
- Рассчитать Retention Rate этих пользователей по дням. Результатом должна быть семидневная retention cohort.

##### Логика расчета Retention Rate:
- Определение когорт по первой дате использования (first_sessions)
- Расчет размера каждой когорты (cohorts)
- Подсчет активных пользователей по дням (retention_counts)
- Расчет процента удержания (retention_rate)

##### Комментарий:
- Долго не мог получить внятные цифры, только после удаления лишних символов в поле 'session_time', получил результат.

### Решение 

In [ ]:
# Загружаем данные
dates = pd.read_csv('C:/Users/Downloads/retention_cohort.csv')

NameError: name 'pd' is not defined

In [3]:
# Знакомимся
dates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2085 entries, 0 to 2084
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   device_id     2085 non-null   object
 1   session_time  2085 non-null   object
dtypes: object(2)
memory usage: 32.7+ KB


In [4]:
# Визуальный осмотр
print(dates)

                             device_id             session_time
0     88741c6b6ba11c1c3c0ad84a41e9b8c6  2019-03-01 00:04:27 UTC
1     386fbfb080e2d7480c82a327548f39fe  2019-03-01 16:29:11 UTC
2     5c68ff813b19a2afc0b5b966c06e03cb  2019-03-01 05:15:59 UTC
3     eeea71c9ef3df15335f4342b3f9b52a6  2019-03-01 14:28:24 UTC
4     ece0296bd2b131e86dc5dd4e2c195fa0  2019-03-01 03:24:11 UTC
...                                ...                      ...
2080  32677be511ae0d408459ad440daa1d99  2019-03-02 05:45:39 UTC
2081  5b2fb4fcfa7130285c8e15b937a2ae24  2019-03-04 11:52:38 UTC
2082  7788456a909d91c522a67005f70a1498  2019-03-01 04:34:14 UTC
2083  434d5be7b893513ebd3f1855488a0da5  2019-03-01 11:39:53 UTC
2084  e77a6be55746e66d979401a0fb02d74b  2019-03-03 13:12:46 UTC

[2085 rows x 2 columns]


In [5]:
# Удаляем ' UTC' из строк с датой
dates['session_time'] = dates['session_time'].str.replace(' UTC', '')

# Проверяем
dates.head()

,device_id,session_time
0,88741c6b6ba11c1c3c0ad84a41e9b8c6,2019-03-01 00:04:27
1,386fbfb080e2d7480c82a327548f39fe,2019-03-01 16:29:11
2,5c68ff813b19a2afc0b5b966c06e03cb,2019-03-01 05:15:59
3,eeea71c9ef3df15335f4342b3f9b52a6,2019-03-01 14:28:24
4,ece0296bd2b131e86dc5dd4e2c195fa0,2019-03-01 03:24:11


In [6]:
# Создаём соединение
conn = sqlite3.connect(':memory:')
dates.to_sql('retention_data', conn, index=False, if_exists='replace')

# Sql-запрос
retention_query = """
-- CTE 1: Находим первую дату сессии для каждого устройства
WITH first_sessions AS (
    SELECT 
        device_id,
        DATE(MIN(session_time)) AS cohort_date  -- Первая дата как дата когорты
    FROM retention_data
    GROUP BY device_id  -- Группируем по устройству
),
-- CTE 2: Формируем когорты по уникальным устройствам
    cohorts AS (
    SELECT 
        cohort_date,
        COUNT(DISTINCT device_id) AS cohort_size  -- Количество устройств в когорте
    FROM first_sessions
    GROUP BY cohort_date  -- Группируем по дате когорты
),
-- CTE 3: Считаем удержание по дням для каждой когорты
    retention_counts AS (
    SELECT 
        fs.cohort_date,
        -- Вычисляем номер дня относительно даты когорты (0 - день первой сессии)
        CAST((julianday(DATE(rd.session_time)) - julianday(fs.cohort_date)) AS INTEGER) AS day_number, -- Для получения целого числа дней
        COUNT(DISTINCT rd.device_id) AS retained_users  -- Количество уникальных вернувшихся устройств
    FROM retention_data as rd
    JOIN first_sessions as fs ON rd.device_id = fs.device_id  -- Соединяем по device_id
    WHERE DATE(rd.session_time) >= fs.cohort_date  -- Только сессии после первой
    GROUP BY fs.cohort_date, day_number  -- Группируем по дате когорты и дню
    HAVING day_number BETWEEN 0 AND 7  -- Фильтруем только первые 7 дней
)
-- Финальный результат с расчетом процента удержания
SELECT 
    c.cohort_date,  -- Дата формирования когорты
    c.cohort_size,  -- Размер когорты
    rc.day_number,  -- День относительно даты когорты (0-6)
    rc.retained_users,  -- Количество оставшихся пользователей
    ROUND(rc.retained_users * 100.0 / c.cohort_size, 2) AS retention_rate -- Расчет процента удержания с округлением
FROM cohorts as c
JOIN retention_counts as rc ON c.cohort_date = rc.cohort_date  -- Соединяем по дате когорты
ORDER BY c.cohort_date, rc.day_number;  -- Сортировка по дате когорты и дню
"""
retention_result = pd.read_sql_query(retention_query, conn)
print("\nИтого семидневная retention cohort:")
print(retention_result)

# Закрываем соединение
conn.close()


Итого семидневная retention cohort:
  cohort_date  cohort_size  day_number  retained_users  retention_rate
0  2019-03-01          862           0             862          100.00
1  2019-03-01          862           1             190           22.04
2  2019-03-01          862           2             122           14.15
3  2019-03-01          862           3              94           10.90
4  2019-03-01          862           4              78            9.05
5  2019-03-01          862           5              75            8.70
6  2019-03-01          862           6              73            8.47


## Задание № 2
- Есть таблица COFFEE_SALES с данными по продажам кофе.
- Вычислить медианное значение суммарных месячных трат постоянных клиентов за каждый месяц.

##### Логика решения:
- monthly_spending - сначала мы группируем данные по клиентам и месяцам, считая сколько каждый клиент потратил в каждом месяце (учитывая скидки).
- constant_customers - находим клиентов, которые делали покупки во всех месяцах. Это те, у кого количество уникальных месяцев равно общему количеству месяцев в данных.
- constant_customers_spending - оставляем только данные по этим постоянным клиентам.
- ranked_spending - для каждого месяца сортируем траты клиентов по возрастанию и нумеруем их. Это нужно для нахождения медианы.
- В финальном запросе мы: для нечетного количества клиентов берем одну среднюю строку, для четного - две средние строки и считаем среднее между ними. 

##### Комментарий:
- С PERCENTILE_CONT код был-бы лаконичней).
- Логика вычисления медианы CTE 5:

Если количество строк нечетное (например, 5):

(5 + 1) / 2 = 3 - берем 3-ю строку

(5 + 2) / 2 = 3.5 → округляется до 3 (так как row_num целое)

В результате WHERE выберет одну строку (3)

Если количество строк четное (например, 6):

(6 + 1) / 2 = 3.5 → округляется до 3 и 4

(6 + 2) / 2 = 4

WHERE выберет две строки (3 и 4)
- Таблицу COFFEE_SALES превратил в csv файл, чтобы видеть промежуточные результаты рассчётов. В ином случае, просто смотря на таюлицу, задачу бы не решил.
- SQLite не поддерживает функцию DATE_TRUNC, изначально, этой функции отдавал предпочтение.
- Вспомнил зачем нужны подзапросы)

### Решение 

In [7]:
# Загружаем данные
df = pd.read_csv('C:/Users/Max Man/Downloads/COFFEE_SALES_TAB.csv')

In [8]:
# Создаём соединение
conn = sqlite3.connect(':memory:')
df.to_sql('COFFEE_SALES', conn, index=False, if_exists='replace')

# Итоговый sql запрос
med_month = """
-- CTE 1. Сначала выделим месяц из даты продажи и посчитаем сумму покупок для каждого клиента по месяцам
WITH monthly_spending AS (
    SELECT 
        CARD_NUMBER,
        SUBSTR(SALE_DTTM, 4, 7) AS month,  -- Извлекаем часть строки с датой, начиная с 4 символа (пропускаем DD/) длиной 7 символов (MM/YYYY)
        SUM(PRICE * (1 - DISCOUNT/100.0)) AS total_spent -- Рассчитываем фактическую цену с учётом скидки
    FROM COFFEE_SALES
    GROUP BY CARD_NUMBER, SUBSTR(SALE_DTTM, 4, 7) -- Группируем данные по клиенту и месяцу
),
-- CTE 2. Считаем количество уникальных месяцев для каждого клиента, сравниваем с общим количеством месяцев в данных.
constant_customers AS (
    SELECT CARD_NUMBER
    FROM monthly_spending
    GROUP BY CARD_NUMBER
    HAVING COUNT(DISTINCT month) = (
        SELECT COUNT(DISTINCT SUBSTR(SALE_DTTM, 4, 7)) 
        FROM COFFEE_SALES
    ) -- Оставляем только тех клиентов, у которых количество месяцев равно общему количеству месяцев в данных
),
-- CTE 3. Отберем только данные по постоянным клиентам
constant_customers_spending AS (
    SELECT 
        month,
        CARD_NUMBER,
        total_spent
    FROM monthly_spending
    WHERE CARD_NUMBER IN (SELECT CARD_NUMBER FROM constant_customers) -- Фильтруем только тех клиентов, которые есть в constant_customers
),
-- CTE 4. Пронумеруем суммы трат в каждом месяце для вычисления медианы
ranked_spending AS (
    SELECT 
        month,
        total_spent,
        ROW_NUMBER() OVER (PARTITION BY month ORDER BY total_spent) AS row_num, -- Сортируем траты клиентов по возрастанию с присвоением номера для каждой строки
        COUNT(*) OVER (PARTITION BY month) AS total_rows -- Считаем общее количество строк в окне
    FROM constant_customers_spending
)
-- CTE 5. Вычислим медиану для каждого месяца
SELECT 
    month,
    AVG(total_spent) AS median
FROM ranked_spending
WHERE 
    row_num = (total_rows + 1) / 2 OR -- Для нечетного количества берем среднее значение
    row_num = (total_rows + 2) / 2 -- Для четного количества берем два средних значения
GROUP BY month
ORDER BY month;
"""  
result = pd.read_sql_query(med_month, conn)
print("\nИтого медианное значение суммарных месячных трат постоянных клиентов за каждый месяц:")
print(result)

# Закрываем соединение
conn.close()


Итого медианное значение суммарных месячных трат постоянных клиентов за каждый месяц:
     month  median
0  01/2021  218.25
1  02/2021  319.45


## Задание № 3

- В приложенной выгрузке дана таблица launch с информацией о запусках приложения(session_id - начало сессии, то есть запуск приложения). Необходимо написать Jupyter-notebook, который формирует файлы ниже:
- Таблица с информацией о том, сколько пришло уникальных пользователей и сколько было запущено сессий с каждого источника, в разрезе по платформам приложения (IOS, Android).
- Таблица с агрегированной и отсортированной информацией, какое кол-во пользователей запустили приложение в каждом городе.

##### Логика решения
- Предобработка данных, один столбец на колонки.
- Группируем уникальных user_id и session_id по source и platform (или по platform).
- Группируем уникальных user_id запустивших приложнеи по region.

### Предобработка данных

In [9]:
# Загружаем данные
launch = pd.read_csv('C:/Users/Max Man/Downloads/launch_test_task.csv')

In [10]:
# Знакомимся
launch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 1 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   user_id;session_id;region;platform;source  960 non-null    object
dtypes: object(1)
memory usage: 7.6+ KB


In [11]:
# Визуальный осмотр
print(launch)

             user_id;session_id;region;platform;source
0    ce438bd29151e78bf09552d958737140;000da8f0-351a...
1    0ee0438b13fe6a3b0b95cd488f6ee1b7;00f9badc-1a72...
2    ccb9a5426b6b76b948fb47e0a4b66725;00fdaeb5-fa6e...
3    a1d16d7ff09a3d256d0a3442ea38f400;01464fac-ea44...
4    4f482f4dc1a2e02ddc9b7e7575f99a46;014dba73-473e...
..                                                 ...
955  c2fe6b21ffeb1b2cc73f212947777fdd;ff16a9d9-e3cc...
956  985de27ac668b6c4f2659cb73133aca8;ff27dc5a-cad6...
957  889b8925e822527eb152a8ab6f2f90be;ff7e5e24-759a...
958  9742973f6cbf0ddf4843cebe7d30b535;ffcf0a83-412c...
959  889b8925e822527eb152a8ab6f2f90be;ffeb9184-3faa...

[960 rows x 1 columns]


In [12]:
# Разделяем столбец на отдельные колонки
split_columns = launch['user_id;session_id;region;platform;source'].str.split(';', expand=True)

# Присваиваем имена 
split_columns.columns = ['user_id', 'session_id', 'region', 'platform', 'source']

# Перезаписываем
launch = split_columns.copy()

In [13]:
launch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     960 non-null    object
 1   session_id  960 non-null    object
 2   region      960 non-null    object
 3   platform    960 non-null    object
 4   source      960 non-null    object
dtypes: object(5)
memory usage: 37.6+ KB


In [14]:
# Визуальный осмотр
display(launch)

,user_id,session_id,region,platform,source
0,ce438bd29151e78bf09552d958737140,000da8f0-351a-40b4-9ffc-3b5708196f0c,Иркутск,2GIS 4 Android,android-app://com.google.android.ext.services
1,0ee0438b13fe6a3b0b95cd488f6ee1b7,00f9badc-1a72-4cf3-8640-c7e862038e39,Барнаул,2GIS 4 Android,android-app://com.whatsapp
2,ccb9a5426b6b76b948fb47e0a4b66725,00fdaeb5-fa6e-4da4-8e33-9df23af9664a,Нур-Султан,2GIS 4 Iphone,https://www.google.com
3,a1d16d7ff09a3d256d0a3442ea38f400,01464fac-ea44-413e-81fd-aece41b925e6,Комсомольск-на-Амуре,2GIS 4 Android,android-app://ru.dublgis.dgismobile
4,4f482f4dc1a2e02ddc9b7e7575f99a46,014dba73-473e-4b20-ad74-d233a57da9e5,Пермь,2GIS 4 Android,android-app://com.viber.voip
...,...,...,...,...,...
955,c2fe6b21ffeb1b2cc73f212947777fdd,ff16a9d9-e3cc-4a7f-a1c5-6a7a78542147,Новосибирск,2GIS 4 Android,android-app://com.whatsapp
956,985de27ac668b6c4f2659cb73133aca8,ff27dc5a-cad6-4a49-8dc4-71dd28b7ca00,Краснодар,2GIS 4 Android,android-app://com.fdelivery.courier
957,889b8925e822527eb152a8ab6f2f90be,ff7e5e24-759a-4127-90bb-d221ccd9cbad,Челябинск,2GIS 4 Android,android-app://ru.citymobil.cargo
958,9742973f6cbf0ddf4843cebe7d30b535,ffcf0a83-412c-42ab-8521-5ef044ae4b9d,Тула,2GIS 4 Iphone,https://www.google.com


### Задача 1 решение

##### Не понял по какому принципу группировать, по source + platform или по platform. Поэтому добавил два варианта решения.

In [15]:
# Группируем по source и platform
launch_group = launch.groupby(['source', 'platform']).agg(
    unique_users=('user_id', 'nunique'),  # Количество уникальных пользователей
    total_sessions=('session_id', 'count') # Общее количество сессий
).reset_index()

# Сортируем для удобства
launch_group = launch_group .sort_values('total_sessions', ascending=False)

# Смотрим результат
display(launch_group )

,source,platform,unique_users,total_sessions
26,android-app://com.whatsapp,2GIS 4 Android,134,199
45,https://www.google.com,2GIS 4 Iphone,108,131
46,https://www.google.com/,2GIS 4 Iphone,90,123
34,android-app://ru.dublgis.dgismobile,2GIS 4 Android,75,80
40,android-app://sinet.startup.inDriver,2GIS 4 Android,15,63
23,android-app://com.taxsee.driver,2GIS 4 Android,13,47
49,https://yandex.ru/,2GIS 4 Iphone,30,41
33,android-app://ru.citymobil.cargo,2GIS 4 Android,1,31
29,android-app://com.yandex.browser,2GIS 4 Android,22,29
27,android-app://com.whatsapp.w4b,2GIS 4 Android,18,29


In [17]:
# Группируем по platform
group_platform = launch.groupby('platform').agg(
    unique_users=('user_id', 'nunique'),  # Количество уникальных пользователей
    session_count=('session_id', 'count')  # Общее количество сессий
).reset_index()

# Результат
display(group_platform)

,platform,unique_users,session_count
0,2GIS 4 Android,362,639
1,2GIS 4 Iphone,234,321


### Задача 2 решение

In [18]:
# Подозреваю, что необходимо рассчитать кол-во уникальных пользователей.
# Группируем по region
group_region = launch.groupby('region')['user_id'].nunique().reset_index()

# Сортируем для удобства
group_region = group_region.sort_values('user_id', ascending=False)

# Снимаем ограничение на максимальное количество строк для вывода
pd.set_option('display.max_rows', None) 

# Результат
display(group_region)

,region,user_id
3,Алматы,70
50,Нур-Султан,47
49,Новосибирск,35
11,Бишкек,30
41,Москва,19
86,Челябинск,18
77,Тюмень,18
33,Красноярск,17
21,Иркутск,14
32,Краснодар,12
